# Scenario 2: A cross-functional team with one data scientist working on an ML model
## MLflow setup:

- tracking server: yes, local server
- backend store: sqlite database
- artifacts store: local filesystem

The experiments can be explored locally by accessing the local tracking server.

To run this example you need to launch the mlflow server locally by running the following command in your terminal:

```bash
mlflow server --backend-store-uri sqlite:///backend.db
```

In [1]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [2]:
print(f"tracking URI: {mlflow.get_tracking_uri()}")

tracking URI: http://127.0.0.1:5000


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1774186887729, experiment_id='0', last_update_time=1774186887729, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")
with mlflow.start_run():

    X, y = load_iris(return_X_y=True)
    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")


2026/03/22 15:41:35 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/03/22 15:41:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/22 15:41:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


default artifacts URI: 'mlflow-artifacts:/1/30935d724d734270887063b075a32f18/artifacts'
🏃 View run rebellious-horse-623 at: http://127.0.0.1:5000/#/experiments/1/runs/30935d724d734270887063b075a32f18
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1774186895820, experiment_id='1', last_update_time=1774186895820, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1774186887729, experiment_id='0', last_update_time=1774186887729, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [6]:
from mlflow.tracking import MlflowClient

client = MlflowClient("http://127.0.0.1:5000")

In [7]:
client.search_registered_models()


[]

In [10]:
run_id = client.search_runs(experiment_ids='1')[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2026/03/22 15:44:15 WARNING mlflow.tracking._model_registry.fluent: Run with id 30935d724d734270887063b075a32f18 has no artifacts at artifact path 'models', registering model based on models:/m-fb68b6a6749945d89fa2018051c0248e instead
2026/03/22 15:44:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1774187055116, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1774187055116, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='30935d724d734270887063b075a32f18', run_link='', source='models:/m-fb68b6a6749945d89fa2018051c0248e', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>